# 06 — Deploy the Fake Review Checker website (Hugging Face Space)

Publishes `space/app.py` and the trained models from Drive `outputs/` to a free Hugging Face Space, giving a permanent website:
`https://huggingface.co/spaces/<your-username>/fake-review-checker`

**Before running:** create a free account at huggingface.co, then create an access token with **Write** permission (Settings → Access Tokens). Paste it into the login box that the second cell shows. No GPU is needed for this notebook.

Re-run this notebook any time the models are retrained to update the website.

In [ ]:
# --- Setup: mount Google Drive (dataset lives there, not in GitHub) ---
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/FakeReviewProject'   # change if your folder differs
DATA_DIR    = f'{PROJECT_DIR}/dataset'                     # train.csv, test.csv
OUT_DIR     = f'{PROJECT_DIR}/outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print(sorted(os.listdir(OUT_DIR)))

In [ ]:
!pip -q install -U huggingface_hub
from huggingface_hub import notebook_login
notebook_login()   # paste your Hugging Face WRITE token in the box

In [ ]:
import json, pickle, shutil
from huggingface_hub import HfApi

SPACE_NAME = 'fake-review-checker'
PRIVATE    = False          # True = only you (and invited members) can open it

api = HfApi()
user = api.whoami()['name']
repo_id = f'{user}/{SPACE_NAME}'
api.create_repo(repo_id, repo_type='space', space_sdk='gradio', private=PRIVATE, exist_ok=True)

# Stage the Space: app code from GitHub + trained models from Drive
STAGE = '/content/space_stage'
shutil.rmtree(STAGE, ignore_errors=True)
!git clone -q --depth 1 https://github.com/AjayRaju-18/ShreyaProject.git /content/repo_src
shutil.copytree('/content/repo_src/space', STAGE)
os.makedirs(f'{STAGE}/models', exist_ok=True)

for f in ['cnn_model.keras', 'cnn_vocab.json', 'cnn_config.json']:
    if os.path.exists(f'{OUT_DIR}/{f}'):
        shutil.copy(f'{OUT_DIR}/{f}', f'{STAGE}/models/{f}')
if os.path.exists(f'{OUT_DIR}/cnn_scaler.pkl'):
    sc = pickle.load(open(f'{OUT_DIR}/cnn_scaler.pkl', 'rb'))   # store as JSON: no sklearn version coupling
    json.dump({'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist()}, open(f'{STAGE}/models/cnn_scaler.json', 'w'))
for m in ['bert_model', 'xlnet_model']:
    if os.path.exists(f'{OUT_DIR}/{m}'):
        shutil.copytree(f'{OUT_DIR}/{m}', f'{STAGE}/models/{m}',
                        ignore=shutil.ignore_patterns('training_args.bin', 'checkpoint-*'))
!du -sh {STAGE}/models/*

In [ ]:
api.upload_folder(folder_path=STAGE, repo_id=repo_id, repo_type='space',
                  commit_message='Deploy Fake Review Checker')
print(f'Website: https://huggingface.co/spaces/{repo_id}')
print('First start takes ~5-10 minutes while the Space installs packages and loads the models.')